## Sub-graphs

### Goal

- [More about Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs)

## State
- Sub-graphs allow you to create and manage different states in different parts of your graph.
- This is particularly useful for multi-agent systems, with teams of agents that each have their own state.
- Let's consider a toy example:
    - I have a system that accepts logs
    - It performs two separate sub-tasks by different agents (summarize logs, find failure modes)
    - I want to perform these two operations in two different sub-graphs.

The most critical thing to understand is how these graphs communicate

- In short, communication is done with over-lapping keys:
  - The Sub-graph can access docs from the parent
  - The parent can access summary/failure_report from sub-graphs



In [1]:
from operator import add
from typing_extensions import TypedDict
from typing import List, Optional, Annotated


In [2]:
class Log(TypedDict):
    id:str
    question: str
    docs: Optional[List[str]]
    answer: str
    grade: Optional[int]
    grader: Optional[str]
    feedback: Optional[str]

## Sub graphs
Here is the failure analysis sub-graph, which uses FailureAnalysisState.

In [3]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

In [4]:
class FailureAnalysisState(TypedDict):
    cleaned_logs: List[Log]
    failures: List[Log]
    fa_summary: str
    processed_logs: List[str]

class FailureAnalysisOutputState(TypedDict):
    fa_summary: str
    processed_logs: List[str]


def get_failures(state):
    """ Get logs that contain a failure """
    cleaned_logs = state["cleaned_logs"]
    failures = [log for log in cleaned_logs if "grade" in log]
    return {"failures": failures}

def generate_summary(state):
    """ Generate summary of failures """
    failures = state["failures"]
    # Add fxn: fa_summary = summarize(failures)
    fa_summary = "Poor quality retrieval of Chroma documentation."
    return {"fa_summary": fa_summary, "processed_logs": [f"failure-analysis-on-log-{failure['id']}" for failure in failures]}
